In [0]:
import mlflow
from pyspark.sql.functions import hour, dayofweek

mlflow.pyspark.ml.autolog()
# Load your clean Silver data
silver_data = spark.read.table("yes_catalog.bronze.stations_silver")

# Extract time-based features
ml_data = silver_data.select(
    "bikes_available", 
    "capacity", 
    hour("ingestion_time").alias("hour"),
    dayofweek("ingestion_time").alias("day_of_week")
).dropna()

2026/04/29 13:14:09 WARNING mlflow.utils.autologging_utils: MLflow pyspark.ml autologging is known to be compatible with 3.2.1 <= pyspark <= 3.5.5, but the installed version is 4.0.0. If you encounter errors during autologging, try upgrading / downgrading pyspark to a compatible version, or try upgrading MLflow.


In [0]:
from pyspark.ml.feature import VectorAssembler

# Define which columns are inputs
assembler = VectorAssembler(
    inputCols=["capacity", "hour", "day_of_week"], 
    outputCol="features"
)

# Transform the data
v_ml_data = assembler.transform(ml_data)

In [0]:
from pyspark.ml.regression import LinearRegression

# Split data
train_df, test_df = v_ml_data.randomSplit([0.8, 0.2], seed=42)

# Define the model (predicting bikes_available)
lr = LinearRegression(featuresCol="features", labelCol="bikes_available")

# Train the model
model = lr.fit(train_df)

print("Model Training Complete!")

2026/04/29 13:14:12 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '652eaa8daf854f379134048db4b8b7c3', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current pyspark.ml workflow
2026/04/29 13:14:12 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Valu

Model Training Complete!


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

# Make predictions on the test set
predictions = model.transform(test_df)

# Calculate error
evaluator = RegressionEvaluator(labelCol="bikes_available", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)

print(f"Model RMSE: {rmse}")
# Interpretation: If RMSE is 2.5, your model is off by about 2.5 bikes on average.

2026/04/29 13:14:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Model RMSE: 8.334716461710151
